In [ ]:
import csv
import json
import os
import re

INPUT_CSV_FILE = 'test.csv'

OUTPUT_JSON_FILE = 'dataset_coco.json'

DATASET_NAME = 'coco'
SPLIT_NAME = 'test' 



def simple_tokenizer(raw_text):
    """간단한 토크나이저: 소문자로 바꾸고 기본적인 구두점을 제거한 후 단어로 분리합니다."""
    text = raw_text.lower()
    text = re.sub(r'[.,!?]', '', text)
    return text.split()

def convert_csv_to_coco_json_final():
    """요청된 경로 처리 방식을 적용하여 CSV를 COCO 형식의 JSON 파일로 변환합니다."""
    
    if not os.path.exists(INPUT_CSV_FILE):
        print(f"오류: 입력 파일 '{INPUT_CSV_FILE}'을 찾을 수 없습니다.")
        return

    print(f"입력 파일 '{INPUT_CSV_FILE}'을 읽어 JSON 데이터를 생성하는 중입니다...")

    final_json_data = {
        'dataset': DATASET_NAME,
        'images': []
    }

    img_id_counter = 0
    sent_id_counter = 0
    coco_id_counter = 500000 

    try:
        with open(INPUT_CSV_FILE, mode='r', encoding='utf-8-sig') as infile:
            reader = csv.DictReader(infile)
            for row in reader:
                raw_img_path = row['img_path']
                
                cleaned_img_path = raw_img_path.lstrip('./') 

                filepath_value = os.path.dirname(cleaned_img_path) 

                filename_value = os.path.basename(cleaned_img_path) 

                captions = [row['A'], row['B'], row['C'], row['D']]

                for caption in captions:
                    sentence_obj = {
                        'tokens': simple_tokenizer(caption),
                        'raw': caption,
                        'imgid': img_id_counter,
                        'sentid': sent_id_counter
                    }

                    image_obj = {
                        'filepath': filepath_value,
                        'sentids': [sent_id_counter],
                        'filename': filename_value,
                        'imgid': img_id_counter,
                        'split': SPLIT_NAME,
                        'sentences': [sentence_obj],
                        'cocoid': coco_id_counter
                    }
                    
                    final_json_data['images'].append(image_obj)
                    
                    img_id_counter += 1
                    sent_id_counter += 1
                    coco_id_counter += 1
                    
    except KeyError as e:
        print(f"오류: CSV 파일에서 '{e}' 열을 찾을 수 없습니다. CSV 파일의 헤더(첫 줄)를 확인해주세요.")
        return
    except Exception as e:
        print(f"CSV 파일 처리 중 오류 발생: {e}")
        return

    print(f"'{OUTPUT_JSON_FILE}' 파일로 최종 데이터를 저장하는 중입니다...")
    try:
        with open(OUTPUT_JSON_FILE, 'w', encoding='utf-8') as outfile:
            json.dump(final_json_data, outfile, indent=2, ensure_ascii=False)
        print(f"\n변환 완료! '{OUTPUT_JSON_FILE}' 파일이 성공적으로 생성되었습니다.")
    except Exception as e:
        print(f"JSON 파일 저장 중 오류 발생: {e}")

if __name__ == '__main__':
    convert_csv_to_coco_json_final()

입력 파일 'train.csv'을 읽어 JSON 데이터를 생성하는 중입니다...
'dataset_coco.json' 파일로 최종 데이터를 저장하는 중입니다...

변환 완료! 'dataset_coco.json' 파일이 성공적으로 생성되었습니다.


In [4]:
from datasets import RetrievalDataset, BinaryClassificationDataset, CrossEntorpyDataset, LanguageModelMultiChoiceDataset
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer("./beit3.spm")

RetrievalDataset.make_coco_dataset_index(
    data_path="./",
    tokenizer=tokenizer,
)

read ./dataset_coco.json
Find 60 images and 240 image-text pairs for karpathy dataset test split !
Write ./coco_retrieval.test.jsonl with 240 items !
